In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ChannelAttention(nn.Module):
    """
    Input -> x: [B, N, C]
    Output -> [B, N, C]
    """
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0, proj_drop=0):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        """x: [B, N, C]"""
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = q.transpose(-2, -1)
        k = k.transpose(-2, -1)
        v = v.transpose(-2, -1)

        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).permute(0, 3, 1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

def test_channel_attention():
    B, N, C = 2, 16, 32
    num_heads = 8
    x = torch.randn(B, N, C)
    model = ChannelAttention(dim=C, num_heads=num_heads)
    with torch.no_grad():
        out = model(x)
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {out.shape}")
    assert out.shape == (B, N, C), "Output shape is incorrect!"
    print("Test passed.")

if __name__ == '__main__':
    test_channel_attention()

Input shape: torch.Size([2, 16, 32])
Output shape: torch.Size([2, 16, 32])
Test passed.


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

class BiasFree_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super(BiasFree_LayerNorm, self).__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.weight = nn.Parameter(torch.ones(normalized_shape))

    def forward(self, x):
        # x: [B, N, C]
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return x / torch.sqrt(sigma + 1e-5) * self.weight

class WithBias_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super(WithBias_LayerNorm, self).__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        # x: [B, N, C]
        mu = x.mean(-1, keepdim=True)
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(sigma + 1e-5) * self.weight + self.bias

class LayerNorm(nn.Module):
    def __init__(self, dim, LayerNorm_type='WithBias'):
        super().__init__()
        if LayerNorm_type == 'BiasFree':
            self.body = BiasFree_LayerNorm(dim)
        else:
            self.body = WithBias_LayerNorm(dim)

    def forward(self, x):
        # x : [B, N, C]
        return self.body(x)

class MSCAttention_N(nn.Module):
    """
    MSCAttention 移植为输入输出均为 (B, N, C) 的形式，等价特征流程
    """
    def __init__(self, dim, num_heads=8, LayerNorm_type='WithBias'):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.norm1 = LayerNorm(dim, LayerNorm_type)
        
        # 用mlp代替卷积（空间信息丢失，但主干结构保持）
        self.project_out = nn.Linear(dim, dim)
        self.linear0_1 = nn.Linear(dim, dim)
        self.linear0_2 = nn.Linear(dim, dim)
        self.linear1_1 = nn.Linear(dim, dim)
        self.linear1_2 = nn.Linear(dim, dim)
        self.linear2_1 = nn.Linear(dim, dim)
        self.linear2_2 = nn.Linear(dim, dim)

    def forward(self, x):
        # x: [B, N, C]
        B, N, C = x.shape

        x1 = self.norm1(x)
        attn_00 = self.linear0_1(x1)
        attn_01 = self.linear0_2(x1)
        attn_10 = self.linear1_1(x1)
        attn_11 = self.linear1_2(x1)
        attn_20 = self.linear2_1(x1)
        attn_21 = self.linear2_2(x1)

        out1 = attn_00 + attn_10 + attn_20
        out2 = attn_01 + attn_11 + attn_21

        out1 = self.project_out(out1)
        out2 = self.project_out(out2)

        # 将特征[head, patch, c]组织，模拟空间轴重组
        assert C % self.num_heads == 0, "C must be divisible by num_heads"
        head_dim = C // self.num_heads
        # N = H * W，不再区分H,W，直接按patch序列处理即可
        k1 = rearrange(out1, "b n (h c) -> b h n c", h=self.num_heads)
        v1 = rearrange(out1, "b n (h c) -> b h n c", h=self.num_heads)
        k2 = rearrange(out2, "b n (h c) -> b h n c", h=self.num_heads)
        v2 = rearrange(out2, "b n (h c) -> b h n c", h=self.num_heads)
        q1 = rearrange(out2, "b n (h c) -> b h n c", h=self.num_heads)
        q2 = rearrange(out1, "b n (h c) -> b h n c", h=self.num_heads)

        q1 = F.normalize(q1, dim=-1); q2 = F.normalize(q2, dim=-1)
        k1 = F.normalize(k1, dim=-1); k2 = F.normalize(k2, dim=-1)

        attn1 = torch.matmul(q1, k1.transpose(-2, -1)) * self.temperature
        attn1 = attn1.softmax(dim=-1)
        out3 = torch.matmul(attn1, v1) + q1
        
        attn2 = torch.matmul(q2, k2.transpose(-2, -1)) * self.temperature
        attn2 = attn2.softmax(dim=-1)
        out4 = torch.matmul(attn2, v2) + q2

        out3 = rearrange(out3, "b h n c -> b n (h c)")
        out4 = rearrange(out4, "b h n c -> b n (h c)")

        out = self.project_out(out3) + self.project_out(out4) + x
        return out

if __name__ == '__main__':
    B, N, C = 2, 1024, 64  # 比如 H=32, W=32, N=1024
    x = torch.randn(B, N, C)
    msca = MSCAttention_N(dim=C)
    out = msca(x)
    print('input_size:', x.size())
    print('output_size:', out.size())

input_size: torch.Size([2, 1024, 64])
output_size: torch.Size([2, 1024, 64])
